# ✈️ MSML 610 – Airport Congestion Project  
## Notebook 3: Model Training (XGBoost)

This notebook trains an **XGBoost classifier** to predict congestion levels  
(**Low**, **Medium**, **High**) for each airport–hour observation.

The input dataset comes from:

data/processed/hourly_congestion.csv

Output model is saved as:

data/models/model.pkl

In [5]:
import pandas as pd
from pathlib import Path

import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from xgboost import XGBClassifier

ROOT = Path.cwd().parents[0]
PROCESSED = ROOT / "data" / "processed"
MODELS = ROOT / "data" / "models"
MODELS.mkdir(parents=True, exist_ok=True)

PROCESSED, MODELS

(PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/processed'),
 PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/models'))

In [7]:
df = pd.read_csv(PROCESSED / "hourly_congestion.csv")
df.head()

,AIRPORT,DATE,HOUR,departures,arrivals,total_flights,congestion_level
0,10135,2015-10-01,6,3,0,3,Low
1,10135,2015-10-01,11,0,2,2,Low
2,10135,2015-10-01,12,3,1,4,Low
3,10135,2015-10-01,15,0,1,1,Low
4,10135,2015-10-01,16,1,2,3,Low


## Feature Selection

We will use the following predictors:

### **Numerical**
- `departures`
- `arrivals`
- `total_flights`
- `HOUR`

### **Categorical**
- `AIRPORT`

In [29]:
# Encode congestion_level into numeric classes
label_map = {"Low": 0, "Medium": 1, "High": 2}

df["label"] = df["congestion_level"].map(label_map)

# Drop rows with missing labels (if any)
df = df.dropna(subset=["label"])

numeric_features = ["departures", "arrivals", "total_flights", "HOUR"]
categorical_features = ["AIRPORT"]

df["AIRPORT"] = df["AIRPORT"].astype(str)

X = df[numeric_features + categorical_features]
y = df["label"].astype(int)

X.head(), y.head()

(   departures  arrivals  total_flights  HOUR AIRPORT
 0           3         0              3     6   10135
 1           0         2              2    11   10135
 2           3         1              4    12   10135
 3           0         1              1    15   10135
 4           1         2              3    16   10135,
 0    0
 1    0
 2    0
 3    0
 4    0
 Name: label, dtype: int64)

In [32]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape

((1037767, 5), (259442, 5))

## Build XGBoost Training Pipeline

The pipeline includes:

1. **One-Hot Encoding** for `AIRPORT`
2. **Passthrough** numerical features
3. **XGBoost Classifier**

In [36]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss"
    ))
])

model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', 'passthrough',
                                                  ['departures', 'arrivals',
                                                   'total_flights', 'HOUR']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['AIRPORT'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None,
                               colsample_bylevel=None, colsample_bynode=None,
                               colsample_bytree=None, device=None,
                               ea...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=200, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [39]:
print("[INFO] Training model...")
model.fit(X_train, y_train)
print("[SUCCESS] Training complete!")

[INFO] Training model...
[SUCCESS] Training complete!


In [41]:
preds = model.predict(X_test)

# Reverse map 0/1/2 → Low/Medium/High
rev_label_map = {0: "Low", 1: "Medium", 2: "High"}
pred_labels = [rev_label_map[p] for p in preds]
true_labels = [rev_label_map[t] for t in y_test]

print("=== Classification Report ===")
print(classification_report(true_labels, pred_labels))

=== Classification Report ===
              precision    recall  f1-score   support

        High       1.00      1.00      1.00      8582
         Low       1.00      1.00      1.00    227204
      Medium       1.00      1.00      1.00     23656

    accuracy                           1.00    259442
   macro avg       1.00      1.00      1.00    259442
weighted avg       1.00      1.00      1.00    259442



In [43]:
## Save Trained Model (model.pkl)

In [45]:
model_path = MODELS / "model.pkl"
joblib.dump(model, model_path)

model_path

PosixPath('/Users/varunparashar/src/umd_classes1/class_project/MSML610/Fall2025/Projects/UmdTask248_Fall2025_XGBoost_Airport_Congestion_Prediction/data/models/model.pkl')

# 🎉 Model Training Completed

Your model is now ready to be used in the Streamlit app: